In [7]:
import cv2
import mediapipe as mp
import numpy as np
import os
import glob
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# ==========================================
# CONFIGURATION
# ==========================================
DATA_DIR = "gesture_videos"  # Folder containing your recorded videos
GESTURES = ["jump", "duck", "left", "right", "neutral"] # List of all your classes
MODEL_TYPE = "RULE_BASED" # Options: "RULE_BASED" or "XGBOOST" (Change this later!)

# ==========================================
# 1. MEDIAPIPE SETUP
# ==========================================
mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils

# Helper functions for Rule-Based Logic
def landmark_xy(landmarks, idx):
    lm = landmarks[idx]
    return lm.x, lm.y, lm.visibility

def center(a, b):
    return ((a[0] + b[0]) * 0.5, (a[1] + b[1]) * 0.5)

def visible(*vs, thr=0.5):
    return all(v >= thr for v in vs)

# ==========================================
# 2. DETECTION MODELS
# ==========================================

def detect_rule_based(pose_landmarks):
    """
    Your existing heuristic/rule-based logic.
    """
    if not pose_landmarks:
        return "neutral"

    lms = pose_landmarks.landmark
    
    # Constants UPDATED based on your calibration results
    LEAN_THRESH = 0.0316
    HANDS_ABOVE_SHOULDERS_DELTA = -0.4119
    CROUCH_TORSO_RATIO = 0.80 # Increased to help catch 'duck' (was 0.62)

    # Keypoints
    NOSE = mp_holistic.PoseLandmark.NOSE
    L_SHO = mp_holistic.PoseLandmark.LEFT_SHOULDER
    R_SHO = mp_holistic.PoseLandmark.RIGHT_SHOULDER
    L_HIP = mp_holistic.PoseLandmark.LEFT_HIP
    R_HIP = mp_holistic.PoseLandmark.RIGHT_HIP
    L_WRIST = mp_holistic.PoseLandmark.LEFT_WRIST
    R_WRIST = mp_holistic.PoseLandmark.RIGHT_WRIST

    nose = landmark_xy(lms, NOSE)
    lsho = landmark_xy(lms, L_SHO)
    rsho = landmark_xy(lms, R_SHO)
    lhip = landmark_xy(lms, L_HIP)
    rhip = landmark_xy(lms, R_HIP)
    lwri = landmark_xy(lms, L_WRIST)
    rwri = landmark_xy(lms, R_WRIST)

    # Visibility check
    if not visible(nose[2], lsho[2], rsho[2], lhip[2], rhip[2], lwri[2], rwri[2]):
        return "neutral"

    sh_cx, sh_cy = center(lsho, rsho)
    hip_cx, hip_cy = center(lhip, rhip)

    dx = sh_cx - hip_cx
    torso = abs(nose[1] - hip_cy)

    wrist_left_y = lwri[1]
    wrist_right_y = rwri[1]
    shoulder_y = (lsho[1] + rsho[1]) * 0.5

    # Logic
    if wrist_left_y < (shoulder_y - HANDS_ABOVE_SHOULDERS_DELTA) and \
       wrist_right_y < (shoulder_y - HANDS_ABOVE_SHOULDERS_DELTA):
        return "jump"

    if torso < CROUCH_TORSO_RATIO:
        return "duck"

    if dx <= -LEAN_THRESH:
        return "left"
    if dx >= LEAN_THRESH:
        return "right"

    return "neutral"

def extract_features_for_ml(pose_landmarks):
    """
    Flattens landmarks into a feature vector for XGBoost/Random Forest.
    Returns: np.array of shape (1, num_features)
    """
    if not pose_landmarks:
        # Return zeros if no person detected (adjust size based on your model)
        return np.zeros((1, 33 * 4)) 
    
    # Extract x, y, z, visibility for all 33 landmarks
    features = []
    for lm in pose_landmarks.landmark:
        features.extend([lm.x, lm.y, lm.z, lm.visibility])
    
    return np.array([features])

def detect_xgboost(pose_landmarks, model):
    """
    Uses the trained ML model to predict.
    """
    # 1. Extract Features
    features = extract_features_for_ml(pose_landmarks)
    
    # 2. Predict (Assuming model is loaded)
    # prediction_index = model.predict(features)[0]
    # return GESTURES[prediction_index]
    
    return "neutral" # Placeholder until you load your model


# ==========================================
# 3. EVALUATION LOOP
# ==========================================

def evaluate_folder():
    y_true = []
    y_pred = []
    
    # Find all video files (mp4, avi, mov) recursively
    video_files = []
    for ext in ['*.mp4', '*.avi', '*.mov']:
        video_files.extend(glob.glob(os.path.join(DATA_DIR, '**', ext), recursive=True))

    print(f"Found {len(video_files)} videos in {DATA_DIR}...")

    # Load Model (If using XGBoost)
    model = None
    if MODEL_TYPE == "XGBOOST":
        # model = joblib.load("my_xgboost_model.pkl") 
        pass

    with mp_holistic.Holistic(
        static_image_mode=False,
        model_complexity=1,
        smooth_landmarks=True
    ) as holistic:

        for video_path in video_files:
            # 1. Determine Ground Truth from filename or folder name
            ground_truth = "unknown"
            filename = os.path.basename(video_path).lower()
            foldername = os.path.basename(os.path.dirname(video_path)).lower()
            
            for g in GESTURES:
                if g in filename or g in foldername:
                    ground_truth = g
                    break
            
            if ground_truth == "unknown":
                print(f"Skipping {filename} (Could not determine gesture from name)")
                continue

            # 2. Process Video
            cap = cv2.VideoCapture(video_path)
            frame_predictions = []
            
            while cap.isOpened():
                ret, frame = cap.read()
                if not ret:
                    break
                
                # Convert to RGB
                image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                results = holistic.process(image)
                
                # 3. Predict
                if MODEL_TYPE == "RULE_BASED":
                    prediction = detect_rule_based(results.pose_landmarks)
                elif MODEL_TYPE == "XGBOOST":
                    prediction = detect_xgboost(results.pose_landmarks, model)
                
                frame_predictions.append(prediction)

            cap.release()

            if not frame_predictions:
                print(f"Warning: No frames processed for {filename}")
                continue

            # 4. Aggregate Predictions (Majority Vote)
            # You can also use specific logic (e.g., if 'jump' appears > 5 frames)
            final_prediction = max(set(frame_predictions), key=frame_predictions.count)
            
            y_true.append(ground_truth)
            y_pred.append(final_prediction)
            
            print(f"Video: {filename} | True: {ground_truth} | Pred: {final_prediction}")

    # ==========================================
    # 4. METRICS
    # ==========================================
    print("\n" + "="*40)
    print(f"EVALUATION REPORT ({MODEL_TYPE})")
    print("="*40)
    
    if len(y_true) > 0:
        print(f"Total Accuracy: {accuracy_score(y_true, y_pred) * 100:.2f}%")
        print("\nClassification Report:")
        print(classification_report(y_true, y_pred, labels=GESTURES, zero_division=0))
        
        print("\nConfusion Matrix:")
        cm = confusion_matrix(y_true, y_pred, labels=GESTURES)
        print(cm)
    else:
        print("No valid video data found to evaluate.")

if __name__ == "__main__":
    evaluate_folder()

Found 15 videos in gesture_videos...


c:\Users\LOQ\OneDrive\Desktop\codes\Applied AI Engineering Lab\Body-Game-Gesture-Detection\AiLab\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Video: duck_1.mp4 | True: duck | Pred: neutral
Video: duck_2.mp4 | True: duck | Pred: neutral
Video: duck_3.mp4 | True: duck | Pred: neutral
Video: jump_1.mp4 | True: jump | Pred: neutral
Video: jump_2.mp4 | True: jump | Pred: neutral
Video: jump_3.mp4 | True: jump | Pred: neutral
Video: left_1.mp4 | True: left | Pred: neutral
Video: left_2.mp4 | True: left | Pred: neutral
Video: left_3.mp4 | True: left | Pred: neutral
Video: neutral_1.mp4 | True: neutral | Pred: neutral
Video: neutral_2.mp4 | True: neutral | Pred: neutral
Video: neutral_3.mp4 | True: neutral | Pred: neutral
Video: right_1.mp4 | True: right | Pred: neutral
Video: right_2.mp4 | True: right | Pred: neutral
Video: right_3.mp4 | True: right | Pred: neutral

EVALUATION REPORT (RULE_BASED)
Total Accuracy: 20.00%

Classification Report:
              precision    recall  f1-score   support

        jump       0.00      0.00      0.00         3
        duck       0.00      0.00      0.00         3
        left       0.00      